# Comparing MOM6 output with in-situ ocean observations (large domain, parallel)

This notebook is the parallel-workflow, larger-domain counterpart of the [Hawaii tutorial](tutorial_MOM6-CL-comparison-Hawaii.ipynb). The steps are the same ones you went through there, but the configuration file is `model2obs.config_tutorial_NWA_parallel.yaml`. The workflow is run with `parallel=True`, which dispatches one worker thread per model file so that the files are processed concurrently.

For this tutorial we will use output from a MOM6 run already stored on NCAR's HPC system, and CrocoLake observation files already in obs_sequence format and also stored on NCAR's HPC system.

## Running the workflow

Running the workflow to interpolate the model is quite simple: create a `WorkflowModelObs` instance using a configuration file, and then call its `run()` method. For this tutorial we will use the configuration file `model2obs.config_tutorial_NWA_parallel.yaml`. A template file to use as reference is also provided at `config_templates/config_template_MOM6.yaml`.

While running, model2obs generates temporary input files that tell DART's `perfect_model_obs` executable where to find MOM6 and CrocoLake data to perform the interpolation.

The difference from the Hawaii tutorial is the `parallel=True` flag passed to `run()`, which processes each model output file in a separate thread. This is useful when the model output is split across multiple files, as is the case here with `in_mom6_par/`.

In [1]:
from model2obs.workflows import WorkflowModelObs

# Create and run workflow to interpolate MOM6 model onto CrocoLake obs space
workflow_crocolake = WorkflowModelObs.from_config_file('model2obs.config_tutorial_NWA_parallel.yaml')
workflow_crocolake.run(clear_output=True, parallel=True)  # use flag clear_output=True if you want to re-run it and automatically clean all previous output

Detailed run log: /glade/work/skygale/crocodile2026/model2obs/tutorial_data/output/tutorial_NW_atlantic_parallel/logs/model2obs.log
Clearing all output folders...
  Clearing folder: /glade/work/skygale/crocodile2026/model2obs/tutorial_data/output/tutorial_NW_atlantic_parallel/out_parquet
  Clearing folder: /glade/work/skygale/crocodile2026/model2obs/tutorial_data/output/tutorial_NW_atlantic_parallel/input_bckp
  Clearing folder: /glade/work/skygale/crocodile2026/model2obs/tutorial_data/output/tutorial_NW_atlantic_parallel/out_trimmed_obs_seq_in
  Clearing folder: /glade/work/skygale/crocodile2026/model2obs/tutorial_data/output/tutorial_NW_atlantic_parallel/out_obs_seq_in
  Clearing folder: netcdf_output
  Clearing folder: /glade/work/skygale/crocodile2026/model2obs/tutorial_data/output/tutorial_NW_atlantic_parallel/logs
  Clearing folder: /glade/work/skygale/crocodile2026/model2obs/tutorial_data/output/tutorial_NW_atlantic_parallel/tmp
  All output folders cleared.

Starting files proc

## Displaying the interactive map

model2obs generates a parquet dataset that contains the values of the CrocoLake observations, the MOM6 model data interpolated onto the observations space, and some basic statistics.

The dataframe by default is stored to `model_obs_df` (as a dask dataframe), and model2obs offers tools to access all data, only the successful interpolations, or only the failed interpolations. For now, let's load only the interpolations that succeeded, which are ~98\% of the total (see output message from previous cell):

In [2]:
good_model_obs_df = workflow_crocolake.get_good_model_obs_df(compute=True) # compute=True triggers the compute of the dask dataframe, returning a pandas dataframe with data loaded in memory
good_model_obs_df.head()                                                   # displays first 5 rows in the dataframe

,time,longitude,latitude,vertical,type,interpolated_model,obs,obs_err_var,interpolated_model_QC,difference,abs_difference,squared_difference,normalized_difference,log_likelihood
0,2010-05-01 16:39:46,293.131000,31.818000,574.855010,ARGO_SALINITY,0.036174,0.036204,0.010,1.0,0.000031,0.000031,9.318380e-10,0.000305,1.383647
1,2010-05-01 09:50:00,315.553009,41.520000,38.593013,ARGO_TEMPERATURE,15.563719,16.106001,0.002,1.0,0.542282,0.542282,2.940696e-01,12.125791,-71.329036
2,2010-05-01 16:39:46,293.131000,31.818000,608.521722,ARGO_TEMPERATURE,15.344611,15.572000,0.002,1.0,0.227389,0.227389,5.170577e-02,5.084573,-10.738076
3,2010-05-01 09:50:00,315.553009,41.520000,1038.493079,ARGO_SALINITY,0.034985,0.034964,0.010,1.0,-0.000021,0.000021,4.516069e-10,-0.000213,1.383647
4,2010-05-01 07:46:00,310.020996,41.506001,1432.521027,ARGO_TEMPERATURE,3.820918,4.178000,0.002,1.0,0.357082,0.357082,1.275078e-01,7.984603,-29.688581


Loading the interactive map to explore the succesfull interpolations is as simple as importing the widget and passing the dataframe to it:

In [3]:
from model2obs.viz import InteractiveWidgetMap
# Create an interactive map widget to visualize model-observation comparisons
# The widget provides controls for selecting variables, observation types, and time ranges
widget = InteractiveWidgetMap(good_model_obs_df)
widget.setup()

## Interpolation errors

The dataframe built by model2obs during `WorkflowModelObs.run()` contains the column `interpolated_model_QC`, which stores information about the quality of the interpolation. The value is set by DART's `perfect_model_obs` program, and you can find more information about it [here](https://docs.dart.ucar.edu/en/stable/assimilation_code/modules/assimilation/quality_control_mod.html) and [here](https://github.com/NCAR/DART/blob/1ddc21f418175fab0768b572866f54a2983dbb66/models/MOM6/model_mod.f90#L119C23-L119C41). In general, for this workflow we want QC≤2, and indeed the method `get_good_model_obs_df()` that we used earlier uses this criterion.

Let's now have a look at the failed interpolations by loading the data and plotting it:

In [4]:
failed_model_obs_df = workflow_crocolake.get_failed_model_obs_df(compute=True) # compute=True triggers the compute of the dask dataframe, returning a pandas dataframe with data loaded in memory
failed_model_obs_df.head()                                                     # displays first 5 rows in the dataframe

,time,longitude,latitude,vertical,type,interpolated_model,obs,obs_err_var,interpolated_model_QC,difference,abs_difference,squared_difference,normalized_difference,log_likelihood
231,2010-05-01 11:24:00,323.000,7.4997,3451.722934,BOTTLE_TEMPERATURE,-888888.0,2.4462,-888888.0,1018.0,888890.4462,888890.4462,7.901262e+11,NaN,NaN
336,2010-05-01 18:40:00,323.499,7.4992,4427.876386,BOTTLE_TEMPERATURE,-888888.0,1.9880,-888888.0,1018.0,888889.9880,888889.9880,7.901254e+11,NaN,NaN
893,2010-05-01 18:29:00,323.499,7.4992,4743.273822,BOTTLE_TEMPERATURE,-888888.0,1.9937,-888888.0,1018.0,888889.9937,888889.9937,7.901254e+11,NaN,NaN
1320,2010-05-01 03:48:00,322.500,7.4998,3939.796016,BOTTLE_TEMPERATURE,-888888.0,2.1360,-888888.0,1018.0,888890.1360,888890.1360,7.901257e+11,NaN,NaN
1455,2010-05-01 03:33:00,322.500,7.4998,4426.512159,BOTTLE_TEMPERATURE,-888888.0,1.8537,-888888.0,1018.0,888889.8537,888889.8537,7.901252e+11,NaN,NaN


In [5]:
widget_failed = InteractiveWidgetMap(failed_model_obs_df)
widget_failed.setup()

We note two things:

1. when an interpolation fails, a value of -888888 is assigned to the interpolated model;
2. from `head()`, we see values of QC greater than 1000.
3. the failed observations are close to the model boundaries (north and east) or to land (west);

The following command gives us the unique values of `model_QC`:

In [6]:
failed_model_obs_df['interpolated_model_QC'].unique()

array([1018.])

We should have only values equal to 1018. Values greater than 1000 are given as 1000 + failure_code, with failure_code from [here](https://github.com/NCAR/DART/blob/1ddc21f418175fab0768b572866f54a2983dbb66/models/MOM6/model_mod.f90#L119C23-L119C41). In this case, the QC flag 1018 indicates that one or more grid points required for the interpolation are not in the basin, so the interpolation cannot be performed. This is consistent with the visual inspection from the map, where we noticed that failed interpolations are closed to model boundary and to the sea/land border.

Finally, if you want to load all the data at once, you can use the following command:

In [7]:
# Load the parquet dataset generated by the workflow above
model_obs_df = workflow_crocolake.get_all_model_obs_df(compute=True) # compute=True triggers the compute of the dask dataframe, returning a pandas dataframe with data loaded in memory
model_obs_df.head() # displays first 5 rows in the dataframe

,time,longitude,latitude,vertical,type,interpolated_model,obs,obs_err_var,interpolated_model_QC,difference,abs_difference,squared_difference,normalized_difference,log_likelihood
0,2010-05-01 16:39:46,293.131000,31.818000,574.855010,ARGO_SALINITY,0.036174,0.036204,0.010,1.0,0.000031,0.000031,9.318380e-10,0.000305,1.383647
1,2010-05-01 09:50:00,315.553009,41.520000,38.593013,ARGO_TEMPERATURE,15.563719,16.106001,0.002,1.0,0.542282,0.542282,2.940696e-01,12.125791,-71.329036
2,2010-05-01 16:39:46,293.131000,31.818000,608.521722,ARGO_TEMPERATURE,15.344611,15.572000,0.002,1.0,0.227389,0.227389,5.170577e-02,5.084573,-10.738076
3,2010-05-01 09:50:00,315.553009,41.520000,1038.493079,ARGO_SALINITY,0.034985,0.034964,0.010,1.0,-0.000021,0.000021,4.516069e-10,-0.000213,1.383647
4,2010-05-01 07:46:00,310.020996,41.506001,1432.521027,ARGO_TEMPERATURE,3.820918,4.178000,0.002,1.0,0.357082,0.357082,1.275078e-01,7.984603,-29.688581
